### 기본환경 셋팅

In [1]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/workspace/exaone35')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)

MODEL_ID = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
SYSTEM_PROMPT = 'You are EXAONE model from LG AI Research, a helpful assistant.'

os.environ['HF_HOME'] = '/workspace/.cache/huggingface'
os.environ['MODEL_ID'] = MODEL_ID
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['DISABLE_DOTENV'] = '1'
os.environ.pop('HF_TOKEN', None)

print('project_dir:', PROJECT_DIR)
print('python:', sys.executable)
print('hf_home:', os.environ['HF_HOME'])
print('model_id:', MODEL_ID)
print('disable_dotenv:', os.environ['DISABLE_DOTENV'])

project_dir: /workspace/exaone35
python: /workspace/.venv/bin/python
hf_home: /workspace/.cache/huggingface
model_id: LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct
disable_dotenv: 1


### GPU 설정 확인

In [2]:
!nvidia-smi
!python3 --version || true

Tue May 26 00:40:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 4500 Blac...    On  |   00000000:CA:00.0 Off |                  Off |
| 30%   26C    P8              9W /  200W |       2MiB /  32623MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### BF16 LoFA 기본으로 사용
- gpu 사용 가능 여부 확인

In [3]:
import importlib.metadata as metadata
import torch

packages = ['torch', 'transformers', 'accelerate', 'datasets', 'peft', 'trl', 'bitsandbytes', 'huggingface_hub']
for name in packages:
    try:
        print(f'{name}:', metadata.version(name))
    except metadata.PackageNotFoundError:
        print(f'{name}: NOT_INSTALLED')

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('torch cuda:', torch.version.cuda)
    print('bf16 supported:', torch.cuda.is_bf16_supported())
    free, total = torch.cuda.mem_get_info(0)
    print('gpu memory free GB:', round(free / 1024**3, 2))
    print('gpu memory total GB:', round(total / 1024**3, 2))

torch: 2.11.0+cu128
transformers: 5.1.0
accelerate: 1.13.0
datasets: 4.8.5
peft: 0.19.1
trl: 1.5.0
bitsandbytes: 0.49.2
huggingface_hub: 1.16.1
cuda available: True
gpu: NVIDIA RTX PRO 4500 Blackwell
torch cuda: 12.8
bf16 supported: True
gpu memory free GB: 31.12
gpu memory total GB: 31.37


### 허깅페이스 로그인
- 대부분 고성능 모델은  허깅페이스에서 사용신청을 하고 권한을 얻은다음 로그인을해야 사용가능

In [4]:
from huggingface_hub import notebook_login

os.environ.pop('HF_TOKEN', None)
notebook_login(skip_if_logged_in=False)

### 우리가 전달한 토큰값으로 로그인이 되었다면 아래 셀의 출력을통해 확인

In [5]:
from huggingface_hub import get_token, whoami

HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)
print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')

logged in user: skn29teacher
token prefix: hf_VJe***


### 셈플데이터 구조 확인
- system : 모델역활 설명
- user : 질문
- assistant : 모델이 학습해야할 정답 답변

In [6]:
import json
from pathlib import Path

TRAIN_FILE = Path('data/train.jsonl')
VALID_FILE = Path('data/valid.jsonl')
VALID_ROLES = {'system', 'user', 'assistant'}

def validate_jsonl(path: Path) -> int:
    count = 0
    with path.open(encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            count += 1
            row = json.loads(line)
            messages = row.get('messages')
            if not isinstance(messages, list) or not messages:
                raise ValueError(f'{path}:{line_no}: messages must be a non-empty list')
            if messages[-1].get('role') != 'assistant':
                raise ValueError(f'{path}:{line_no}: last message must be assistant')
            roles = [m.get('role') for m in messages]
            if 'user' not in roles or 'assistant' not in roles:
                raise ValueError(f'{path}:{line_no}: user and assistant messages are required')
            for idx, message in enumerate(messages):
                if message.get('role') not in VALID_ROLES:
                    raise ValueError(f'{path}:{line_no}: invalid role at message {idx}')
                if not isinstance(message.get('content'), str) or not message['content'].strip():
                    raise ValueError(f'{path}:{line_no}: empty content at message {idx}')
    return count

print('train rows:', validate_jsonl(TRAIN_FILE))
print('valid rows:', validate_jsonl(VALID_FILE))
print('validation: OK')

train rows: 30
valid rows: 5
validation: OK


### 데이터 예시 보기

In [7]:
with TRAIN_FILE.open(encoding='utf-8') as f:
    first_sample = json.loads(next(f))

first_sample

{'messages': [{'role': 'system',
   'content': 'You are EXAONE model from LG AI Research, a helpful assistant.'},
  {'role': 'user', 'content': '한국어 형태소 분석이 왜 중요한가요?'},
  {'role': 'assistant',
   'content': '한국어는 조사와 어미가 단어에 붙어 의미와 문법 기능을 만드는 교착어입니다. 형태소 분석을 하면 문장을 의미 단위로 나누고 품사와 어절 관계를 파악할 수 있어 검색, 챗봇, 번역, 감성 분석 같은 자연어 처리 작업의 정확도를 높일 수 있습니다.'}]}

### 데이터셋 로드(만들기)

In [8]:
from datasets import load_dataset

raw_dataset = load_dataset('json', data_files={
    'train': str(TRAIN_FILE),
    'validation': str(VALID_FILE),
})

raw_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 30
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 5
    })
})

### 모델 로드

In [9]:
from transformers import AutoTokenizer

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print('selected dtype:', DTYPE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print('vocab size:', len(tokenizer))
print('pad token:', tokenizer.pad_token, tokenizer.pad_token_id)
print('eos token:', tokenizer.eos_token, tokenizer.eos_token_id)

selected dtype: torch.bfloat16


config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

configuration_exaone.py:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

vocab size: 102400
pad token: [PAD] 0
eos token: [|endofturn|] 361


In [10]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
model.eval()

first_device = next(model.parameters()).device
print('loaded:', True)
print('first parameter device:', first_device)
if torch.cuda.is_available():
    print('memory allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))
    print('memory reserved GB:', round(torch.cuda.memory_reserved() / 1024**3, 3))

`torch_dtype` is deprecated! Use `dtype` instead!


modeling_exaone.py:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json:   0%|          | 0.00/22.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

loaded: True
first parameter device: cuda:0
memory allocated GB: 4.48
memory reserved GB: 4.969


### 원본 모델 추론

In [11]:
prompt = '한국어 형태소 분석이 왜 중요한지 짧게 설명해줘.'
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': prompt},
]

# 최신 Transformers에서는 apply_chat_template가 BatchEncoding을 반환할 수 있습니다.
# 그래서 Tensor 하나를 generate에 넘기지 않고, input_ids/attention_mask를 dict로 풀어 전달합니다.
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
).to(first_device)

inputs = dict(encoded)
prompt_length = inputs['input_ids'].shape[-1]

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0][prompt_length:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(response)

한국어 형태소 분석은 문장의 의미를 정확하게 이해하는 데 핵심적입니다. 형태소 분석을 통해 단어를 그 기본 의미 단위인 형태소로 분해함으로써:

1. **의미 구분**: 복합어와 혼용어를 명확히 구분하여 단어의 본래 의미를 정확히 파악할 수 있습니다.
2. **문맥 이해**: 단어의 역할(어미, 접두사 등)을 파악해 문맥에 따른 의미 변화를 이해할 수 있습니다.
3. **자연어 처리**: 기계 학습 모델이 한국어의 복잡한 구조를 효과적으로 학습하고 처리할 수 있게 돕습니다.

이러한 분석은 번역, 정보 검색, 챗봇 대화 등 다양한 자연어 처리 작업의 정확성을 크게 향상시킵니다.


학습전 gpu메모리 정리

In [12]:
import gc
del encoded
del inputs
del output_ids
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    print('memory allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))
    print('memory reserved GB:', round(torch.cuda.memory_reserved() / 1024**3, 3))

NameError: name 'gc' is not defined

### 학습 설정값 정의

In [ ]:
OUTPUT_DIR = Path('outputs/exaone35-2.4b-koqa-lora')
MAX_SEQ_LENGTH = 2048
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# EXAONE의 선형층 이름입니다. all-linear를 쓰면 lm_head까지 포함될 수 있어
# tied embedding 구조에서 PEFT 호환 문제가 생길 수 있으므로 명시적으로 지정합니다.
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'c_fc_0', 'c_fc_1', 'c_proj']

print('output_dir:', OUTPUT_DIR)
print('max_seq_length:', MAX_SEQ_LENGTH)
print('epochs:', NUM_TRAIN_EPOCHS)
print('lora targets:', LORA_TARGET_MODULES)

### 학습용 토크나이징 함수

SFT 학습에서는 assistant 답변 부분만 loss를 계산하도록 만드는 것이 중요합니다.

코드에서 하는 일:

- 전체 대화를 모델 입력 문자열로 만듭니다.
- assistant 답변 직전까지의 prompt 길이를 계산합니다.
- prompt 영역의 label은 `-100`으로 마스킹해 loss 계산에서 제외합니다.
- assistant 답변 영역만 학습 대상으로 남깁니다.

In [ ]:
def tokenize_chat_example(example):
    messages = example['messages']
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )

    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )
    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )

    input_ids = full['input_ids']
    labels = input_ids.copy()
    prompt_length = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_length] = [-100] * prompt_length

    return {
        'input_ids': input_ids,
        'attention_mask': full['attention_mask'],
        'labels': labels,
    }

tokenized_dataset = raw_dataset.map(
    tokenize_chat_example,
    remove_columns=raw_dataset['train'].column_names,
    desc='Tokenizing chat data',
)

tokenized_dataset

### Data collator 정의

각 샘플은 길이가 다르므로 batch를 만들 때 padding이 필요합니다.

코드에서 하는 일:

- `input_ids`와 `attention_mask`를 tokenizer가 padding합니다.
- `labels`는 padding 위치를 `-100`으로 채워 loss 계산에서 제외합니다.

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class CausalDataCollator:
    tokenizer: Any

    def __call__(self, features):
        labels = [feature.pop('labels') for feature in features]
        batch = self.tokenizer.pad(features, padding=True, return_tensors='pt')
        max_length = batch['input_ids'].shape[1]
        padded_labels = []
        for label in labels:
            padded_labels.append(label + [-100] * (max_length - len(label)))
        batch['labels'] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

collator = CausalDataCollator(tokenizer)
print('collator ready')

### LoRA 학습 모델 구성

여기서 base model을 다시 로드하고 LoRA adapter를 붙입니다.

코드에서 하는 일:

- EXAONE base model을 BF16으로 GPU에 로드합니다.
- gradient checkpointing을 켜서 메모리를 줄입니다.
- PEFT의 `LoraConfig`로 LoRA 학습 대상을 설정합니다.
- 전체 파라미터 중 실제 학습되는 파라미터 수를 출력합니다.

In [ ]:
from peft import LoraConfig, get_peft_model
import types


def patch_exaone_embedding_access(model):
    """PEFT가 EXAONE의 입력/출력 embedding을 찾을 수 있게 보정합니다."""
    if not hasattr(model, 'transformer') or not hasattr(model.transformer, 'wte'):
        return model

    def get_input_embeddings_for_lm(self):
        return self.transformer.wte

    def set_input_embeddings_for_lm(self, value):
        self.transformer.wte = value

    def get_output_embeddings_for_lm(self):
        return self.lm_head

    def set_output_embeddings_for_lm(self, value):
        self.lm_head = value

    def get_input_embeddings_for_base(self):
        return self.wte

    def set_input_embeddings_for_base(self, value):
        self.wte = value

    model.get_input_embeddings = types.MethodType(get_input_embeddings_for_lm, model)
    model.set_input_embeddings = types.MethodType(set_input_embeddings_for_lm, model)
    model.get_output_embeddings = types.MethodType(get_output_embeddings_for_lm, model)
    model.set_output_embeddings = types.MethodType(set_output_embeddings_for_lm, model)
    model.transformer.get_input_embeddings = types.MethodType(get_input_embeddings_for_base, model.transformer)
    model.transformer.set_input_embeddings = types.MethodType(set_input_embeddings_for_base, model.transformer)
    return model


train_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map={'': 0} if torch.cuda.is_available() else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)

# gradient_checkpointing_enable과 PEFT 모두 get_input_embeddings를 사용하므로
# 두 기능을 켜기 전에 EXAONE embedding 접근자를 먼저 보정합니다.
train_model = patch_exaone_embedding_access(train_model)
train_model.config.use_cache = False
train_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)

train_model = get_peft_model(train_model, lora_config)
train_model.print_trainable_parameters()

if torch.cuda.is_available():
    print('memory allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))

### Trainer 설정

Transformers `Trainer`로 학습 루프를 구성합니다.

코드에서 하는 일:

- batch size와 gradient accumulation을 설정합니다.
- BF16 학습을 켭니다.
- validation 평가는 일정 step마다 수행합니다.
- 학습 로그는 외부 서비스로 보내지 않고 노트북에만 출력합니다.

In [ ]:
import inspect
from transformers import Trainer, TrainingArguments

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_strategy='epoch',
    save_total_limit=2,
    bf16=DTYPE == torch.bfloat16,
    fp16=DTYPE == torch.float16,
    gradient_checkpointing=True,
    optim='adamw_torch',
    report_to='none',
    remove_unused_columns=False,
)

signature = inspect.signature(TrainingArguments.__init__)
if 'eval_strategy' in signature.parameters:
    training_kwargs['eval_strategy'] = 'steps'
else:
    training_kwargs['evaluation_strategy'] = 'steps'
training_kwargs['eval_steps'] = 10

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=collator,
)

print('trainer ready')